In [ ]:
import os

config = {
    'bootstrap.servers': os.environ['KAFKA_BOOTSTRAP_SERVERS'],
    'security.protocol': 'SASL_SSL',
    'sasl.mechanisms': 'PLAIN',
    'sasl.username': os.environ['KAFKA_API_KEY'],
    'sasl.password': os.environ['KAFKA_API_SECRET'],
}


In [ ]:
!pip install confluent-kafka

In [ ]:
import requests

In [ ]:
# Fetch data from Alpha Vantage API
url = 'https://www.alphavantage.co/query?function=TIME_SERIES_INTRADAY&symbol=IBM&interval=5min&apikey=demo'
response = requests.get(url)
data = response.json()

In [ ]:
import os

config = {
    'bootstrap.servers': os.environ['KAFKA_BOOTSTRAP_SERVERS'],
    'security.protocol': 'SASL_SSL',
    'sasl.mechanisms': 'PLAIN',
    'sasl.username': os.environ['KAFKA_API_KEY'],
    'sasl.password': os.environ['KAFKA_API_SECRET'],
}


In [ ]:
#PRODUCER
from confluent_kafka import Producer
producer = Producer(**config)



In [ ]:
# Delivery report callback for produced messages
import json
def delivery_report(err, msg):
    """ Called once for each message produced to indicate delivery result.
        Triggered by poll() or flush(). """
    if err is not None:
        print('Message delivery failed: {}'.format(err))
    else:
        print('Message delivered to {} [{}]'.format(msg.topic(), msg.partition()))

# Produce messages to the 'users' topic
topic = 'topic_0'

try:
    for timestamp, values in data['Time Series (5min)'].items():
        record_key = timestamp
        record_value = json.dumps(values)
        producer.produce(topic, key=record_key, value=record_value, callback=delivery_report)
        # Wait up to 1 second for events. Callbacks will be invoked during this method call if the message is acknowledged.
        producer.poll(1)
except Exception as e:
    print(f'Failed to produce message: {e}')

# Wait for any outstanding messages to be delivered and delivery report callbacks to be triggered.
producer.flush()